# Clase 36 - Notebook 4 - Toma de Decisiones Multicriterio (TOPSIS y DFA)

En este notebook aplicamos la metodología **TOPSIS** para seleccionar una solución de compromiso sobre el frente de Pareto obtenido.


### ⚙️ Paso 0: Configuración Automática del Entorno (Google Colab)
Si estás ejecutando este cuaderno en **Google Colab**, ejecuta la siguiente celda una sola vez al inicio de la sesión para clonar automáticamente el repositorio e instalar todas las dependencias requeridas.


In [ ]:
# --- Preámbulo Universal para Google Colab y Entornos Locales ---
import os, sys

if 'google.colab' in sys.modules:
    REPO_DIR = '/content/DAII-SprayDrying'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/felipehuerta17/DAII-SprayDrying.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    %pip install -q numpy scipy pandas matplotlib pymoo casadi
    print('✅ Entorno de Google Colab configurado con éxito.')
else:
    print('✅ Ejecutando en entorno local.')


## Algoritmo de Decisión TOPSIS y Optimización DFA


In [ ]:
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from spraydrylib.desirability import build_obj2, desirability_min, overall_desirability
from spraydrylib.mcdm import topsis, build_comparison_dataframe
from scipy import optimize as opt

# 1) Cargar frente NSGA-II más reciente
nsga_files = sorted(glob.glob("./outputs/front_nsga2_*.csv"))
if not nsga_files:
    raise FileNotFoundError("Ejecuta 02_NSAG2_Pareto para generar ./outputs/front_nsga2_*.csv")

csv_nsga = nsga_files[-1]
df_nsga  = pd.read_csv(csv_nsga)
X = df_nsga[["G_kg_h", "rd_m"]].to_numpy()
F = df_nsga[["Energia_kW", "Xo_prom_ultimos"]].to_numpy()

# 2) TOPSIS para 3 combinaciones de pesos
w0=[0.8,0.2]; w1=[0.2,0.8]; w2=[0.5,0.5]; w=[w0,w1,w2]
max_obj=[]; min_obj=[0,1]

idx = [topsis(F, wi, max_obj, min_obj)[0] for wi in w]
y_TOPSIS = np.array([F[i] for i in idx])
x_TOPSIS = np.array([X[i] for i in idx])

# 3) DFA para los mismos pesos
obj2, _ = build_obj2(tf=400.0, n_steps=600)
bounds  = opt.Bounds([462.0, 3.5e-5], [858.0, 6.5e-5])
L = np.array([16.0, 0.045]); U = np.array([30.0, 0.070])

def D_from_f(f, ww):
    d1 = desirability_min(f[0], L[0], U[0])
    d2 = desirability_min(f[1], L[1], U[1])
    return overall_desirability([d1, d2], ww)

def Obj_DFA(x, ww): return -D_from_f(obj2(x), ww)

x_DFA2 = []; y_DFA2 = []
for wi in w:
    r = opt.differential_evolution(lambda x: Obj_DFA(x, wi), bounds=bounds, seed=42, polish=True, maxiter=50)
    x_DFA2.append(r.x); y_DFA2.append(obj2(r.x))
x_DFA2 = np.array(x_DFA2)
y_DFA2 = np.array(y_DFA2)


In [ ]:
# Gráfica de selección combinada
plt.figure(figsize=(7.5,5.0))
plt.scatter(F[:,0],F[:,1],s=28,color="0.65",edgecolor="black",alpha=0.5,label="Frente de Pareto (NSGA-II)")
for pt,m,lab in zip(y_TOPSIS, ["*","D","s"], [f"TOPSIS {tuple(w0)}",f"TOPSIS {tuple(w1)}",f"TOPSIS {tuple(w2)}"]):
    plt.scatter([pt[0]],[pt[1]],s=180,marker=m,edgecolor="black",color="#1f77b4",label=lab)
for pt,m,lab in zip(y_DFA2, ["X","^","P"], [f"DFA {tuple(w0)}",f"DFA {tuple(w1)}",f"DFA {tuple(w2)}"]):
    plt.scatter([pt[0]],[pt[1]],s=180,marker=m,edgecolor="black",color="#ff7f0e",label=lab)
plt.xlabel("Gasto energético (kW)"); plt.ylabel("Contenido de agua en el droplet (kg/kg)")
plt.title("Selección por TOPSIS y DFA (mismos pesos)"); plt.legend(ncol=2); plt.tight_layout()
stamp=time.strftime("%Y%m%d-%H%M%S"); plt.savefig(f"./outputs/topsis_vs_dfa_{stamp}.png",dpi=200); plt.savefig(f"./outputs/topsis_vs_dfa_{stamp}.svg")
plt.show()


In [ ]:
# Tabla comparativa de decisiones
rows = []
for wi, xi, fi in zip(w, x_TOPSIS, y_TOPSIS):
    rows.append({"metodo": "NSGAII-TOPSIS", "pesos": f"{wi[0]} - {wi[1]}", "G_kg_h": xi[0], "rd_m": xi[1], "energia_kw": fi[0], "humedad": fi[1]})
for wi, xi, fi in zip(w, x_DFA2, y_DFA2):
    rows.append({"metodo": "DFA", "pesos": f"{wi[0]} - {wi[1]}", "G_kg_h": xi[0], "rd_m": xi[1], "energia_kw": fi[0], "humedad": fi[1]})

tabla = build_comparison_dataframe(rows)
tabla_path = f"./outputs/topsis_dfa_tabla_{stamp}.csv"
tabla.to_csv(tabla_path, index=False)
print("Tabla guardada en:", tabla_path)
display(tabla)
